# Aula 02 — Processamento de Linguagem Natural

Células na ordem dos slides; algumas dependem de variáveis de células anteriores.

## Slide 13 — REGEX EM PYTHON

*Bloco 01*

In [ ]:
import re
# Padrões básicos úteis em pré-processamento de texto
texto = "<p>Adorei!! https://link.com vc tem que ver isso :)</p>"
# Remover tags HTML
limpo = re.sub(r'<[^>]+>', ' ', texto)
# 'Adorei!! https://link.com vc tem que ver isso :)'
# Remover URLs
limpo = re.sub(r'http\S+|www\.\S+', ' ', limpo)
# 'Adorei!!  vc tem que ver isso :)'
# Remover e-mails
limpo = re.sub(r'\S+@\S+', ' ', limpo)
# Manter apenas letras (incluindo acentuadas) e espaços
limpo = re.sub(r'[^a-záàâãäéèêëíìîïóòôõöúùûüç\s]', ' ', limpo)
# 'Adorei   vc tem que ver isso  '
# Colapsar espaços múltiplos
limpo = re.sub(r'\s+', ' ', limpo).strip()
# 'Adorei vc tem que ver isso'

## Slide 14 — SETUP DO AMBIENTE

*Bloco 02*

In [ ]:
import re
import unicodedata
import nltk
import spacy
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
# Recursos NLTK em português
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('rslp', quiet=True)
# Modelo spaCy em português
# python -m spacy download pt_core_news_lg
nlp = spacy.load('pt_core_news_lg')
# Corpus exemplo: reviews sujas reais
reviews = [
    "<p>ADOREI o produto!! vc precisa cmprar tbm 😍</p>",
    "Pessimo!! veio quebrado >:( https://reclame.com",
    "Mt bom mesmo, entrega rapida! 5 estrelas pra esse vendedor.",
    "Não gostei :( voltei o item, atendimento foi pessimo tbm.",
]

## Slide 15 — LIMPEZA: HTML, URLS E CONTRAÇÕES

*Bloco 03*

In [ ]:
# Mapa de contrações comuns em PT-BR informal
CONTRACTIONS = {
    'vc': 'você', 'tb': 'também', 'tbm': 'também',
    'pq': 'por que', 'obg': 'obrigado', 'blz': 'beleza',
    'td': 'tudo', 'msm': 'mesmo', 'vlw': 'valeu',
    'n': 'não', 'naum': 'não', 'cmprar': 'comprar',
    'mt': 'muito', 'pra': 'para'
}
def limpar_texto(texto):
    texto = texto.lower()                                  # 1. Lowercase
    texto = re.sub(r'<[^>]+>', ' ', texto)                 # 2. HTML
    texto = re.sub(r'http\S+|www\.\S+', ' ', texto)        # 3. URLs
    texto = re.sub(r'\S+@\S+', ' ', texto)                 # 4. E-mails
    # 5. Expande contrações
    palavras = texto.split()
    palavras = [CONTRACTIONS.get(p, p) for p in palavras]
    return ' '.join(palavras)
print(limpar_texto(reviews[0]))
# Saída: 'adorei o produto!! você precisa comprar também 😍'

## Slide 18 — TRATAMENTO DE ACENTOS DECISÃO SENSÍVEL AO DOWNSTREAM

*Bloco 04*

In [ ]:
import unicodedata
def remove_acentos(texto):
    """Remove acentos: 'maravilhoso' → 'maravilhoso'.
    Útil para busca; perigoso para NER."""
    nfkd = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))
# Exemplos
print(remove_acentos('São Paulo'))      # 'Sao Paulo'
print(remove_acentos('experiência'))    # 'experiencia'
print(remove_acentos('coração'))        # 'coracao'

## Slide 20 — PIPELINE COMPLETO: LIMPAR_NORMALIZAR() A FUNÇÃO QUE COSTURA TUDO

*Bloco 05*

In [ ]:
def limpar_normalizar(texto, remove_acentos=False):
    """Pipeline modular para texto em PT-BR."""
    if not isinstance(texto, str):
        return ''
    texto = texto.lower()                                       # 1
    texto = re.sub(r'<[^>]+>', ' ', texto)                      # 2
    texto = re.sub(r'http\S+|www\.\S+', ' ', texto)             # 3
    texto = re.sub(r'\S+@\S+', ' ', texto)                      # 4
    palavras = [CONTRACTIONS.get(p, p) for p in texto.split()]  # 5
    texto = ' '.join(palavras)
    texto = re.sub(r'[^a-záàâãäéèêëíìîïóòôõöúùûüç\s.,!?]', ' ', texto)  # 6
    if remove_acentos:                                          # 7
        nfkd = unicodedata.normalize('NFKD', texto)
        texto = ''.join(c for c in nfkd if not unicodedata.combining(c))
    texto = re.sub(r'\s+', ' ', texto).strip()                  # 8
    return texto
# Teste
sujo = '<p>Adorei o produto!! https://exemplo.com VC PRECISA TBM!!!</p>'
print(limpar_normalizar(sujo))
# 'adorei o produto !! você precisa também !!!'

## Slide 30 — FAMÍLIA 1: SPLIT() INGÊNUO O BASELINE DIDÁTICO

*Bloco 06*

In [ ]:
# split() simples — divide só por espaços
texto = "O Sr. Silva, hoje, tem R$ 1.250,99 no banco!"
tokens = texto.split()
print(tokens)
# ['O', 'Sr.', 'Silva,', 'hoje,', 'tem', 'R$', '1.250,99', 'no', 'banco!']
# Problemas:
# - 'Silva,' (com vírgula) ≠ 'Silva' → tokens DIFERENTES no vocabulário
# - 'banco!' (com !) ≠ 'banco' → tokens DIFERENTES
# - 'hoje,' não separa a vírgula
# Útil só como BASELINE didático.
# Em produção: NUNCA use split() puro.

## Slide 31 — FAMÍLIA 2A: NLTK WORD_TOKENIZE ROBUSTO, COM SUPORTE A PORTUGUÊS

*Bloco 07*

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize
texto = "O Sr. Silva, hoje, tem R$ 1.250,99 no banco! Adorei."
# Tokenização de palavras (com language='portuguese')
tokens = word_tokenize(texto, language='portuguese')
print(tokens)
# ['O', 'Sr.', 'Silva', ',', 'hoje', ',', 'tem', 'R$',
#  '1.250,99', 'no', 'banco', '!', 'Adorei', '.']
# ✅ 'Sr.' reconhecido como abreviação
# ✅ Vírgulas separadas como tokens
# ✅ Número 1.250,99 mantido inteiro
# Tokenização de FRASES
frases = sent_tokenize(texto, language='portuguese')
print(frases)
# ['O Sr. Silva, hoje, tem R$ 1.250,99 no banco!', 'Adorei.']

## Slide 32 — FAMÍLIA 2B: SPACY O PADRÃO DA INDÚSTRIA MODERNA

*Bloco 08*

In [ ]:
import spacy
nlp = spacy.load('pt_core_news_lg')
texto = "O Sr. Silva, hoje, tem R$ 1.250,99 no banco!"
# spaCy: tokenização + POS + lema + dependência tudo de uma vez
doc = nlp(texto)
for tok in doc:
    print(f"{tok.text:<12} {tok.lemma_:<12} {tok.pos_}")
# Saída:
# O           o           DET
# Sr.         Sr.         PROPN
# Silva       Silva       PROPN
# ,           ,           PUNCT
# hoje        hoje        ADV
# ...
# 1.250,99    1.250,99    NUM
# no          em o        ADP
# banco       banco       NOUN
# !           !           PUNCT
# 🚀 Vantagem: tokens já vêm com lema, POS, dependência!

## Slide 33 — SENTENCE TOKENIZATION QUEBRANDO TEXTO EM FRASES

*Bloco 09*

In [ ]:
# Tokenização de FRASES (não palavras)
from nltk.tokenize import sent_tokenize
texto = ("Adorei o produto. Chegou rápido! "
        "Recomendo a todos. Mas o preço foi alto...")
frases = sent_tokenize(texto, language='portuguese')
for i, f in enumerate(frases, 1):
    print(f'{i}. {f}')
# 1. Adorei o produto.
# 2. Chegou rápido!
# 3. Recomendo a todos.
# 4. Mas o preço foi alto...
# Em spaCy é automático
for sent in nlp(texto).sents:
    print(sent.text)

## Slide 37 — BPE: BYTE-PAIR ENCODING O ALGORITMO DOS TRANSFORMERS

*Bloco 10*

In [ ]:
# Exemplo simplificado de BPE em ação
# Vocabulário inicial: caracteres individuais
# 'l', 'o', 'w', 'e', 'r', ...
# Iteração 1: 'l' + 'o' aparece muito → mescla → 'lo'
# Iteração 2: 'lo' + 'w' aparece muito → 'low'
# Iteração 3: 'e' + 'r' aparece muito → 'er'
# Após várias iterações: vocabulário tem subwords
# como 'low', 'er', 'est', 'ing', '##ing'
# Resultado para 'lowest':
# ['low', 'est']
# Para palavra rara 'newer':
# ['n', 'ewer'] ou ['new', '##er']

## Slide 40 — CURIOSIDADE POR QUE GPT E BERT TOKENIZAM DE FORMA ESTRANHA

*Bloco 11*

In [ ]:
# Tokenizer do BERTimbau (BERT em PT-BR)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')
print(tok.tokenize('correríamos pela praia'))
# ['corre', '##ría', '##mos', 'pela', 'praia']
print(tok.tokenize('inconstitucionalissimamente'))
# ['in', '##cons', '##ti', '##tu', '##cion', '##ali', '##ss', '##im', '##amente']

## Slide 50 — RSLP EM AÇÃO O STEMMER BRASILEIRO

*Bloco 12*

In [ ]:
from nltk.stem import RSLPStemmer
import nltk
nltk.download('rslp', quiet=True)
stemmer = RSLPStemmer()
exemplos = ['correndo', 'correu', 'correriam',
            'maravilhoso', 'felicidades', 'menino',
            'meninas', 'casinha', 'gatos']
for palavra in exemplos:
    print(f'{palavra:<15} → {stemmer.stem(palavra)}')
# Saída:
# correndo        → corr
# correu          → corr
# correriam       → corr
# maravilhoso     → maravilh
# felicidades     → felic
# menino          → menin
# meninas         → menin
# casinha         → casinh
# gatos           → gat

## Slide 51 — LEMMATIZATION COM SPACY MAIS LENTO, MAIS PRECISO

*Bloco 13*

In [ ]:
import spacy
nlp = spacy.load('pt_core_news_lg')
texto = 'As crianças estavam correndo felizes pela praia maravilhosa.'
doc = nlp(texto)
for token in doc:
    print(f'{token.text:<14} → {token.lemma_}')
# Saída:
# As             → o
# crianças       → criança
# estavam        → estar
# correndo       → correr
# felizes        → feliz
# pela           → por o
# praia          → praia
# maravilhosa    → maravilhoso
# .              → .
# 🎯 Repare: lemmatization PRESERVA palavras válidas
# 'correr' (não 'corr'), 'maravilhoso' (não 'maravilh')

## Slide 54 — O QUE SÃO STOP WORDS? PALAVRAS FREQUENTES COM POUCO CONTEÚDO SEMÂNTICO

*Bloco 14*

In [ ]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)
STOP_WORDS_PT = set(stopwords.words('portuguese'))
print(f'Total: {len(STOP_WORDS_PT)} stop words')
# Total: 207 stop words
print('Algumas:', list(STOP_WORDS_PT)[:15])
# ['o', 'a', 'os', 'as', 'de', 'do', 'da', 'em', 'no', 'na',
#  'que', 'um', 'uma', 'para', 'com', ...]

## Slide 57 — PIPELINE NLP FINAL TUDO COSTURADO EM UMA FUNÇÃO

*Bloco 15*

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer
STOP_WORDS_PT = set(stopwords.words('portuguese'))
stemmer = RSLPStemmer()
def pipeline_nlp(textos: list[str]) -> list[list[str]]:
    """Lista de textos PT → lista de listas de tokens limpos."""
    resultado = []
    for texto in textos:
        texto = limpar_normalizar(texto)                              # 1
        tokens = word_tokenize(texto, language='portuguese')          # 2
        tokens = [t for t in tokens                                   # 3
                  if t not in STOP_WORDS_PT and len(t) > 2]
        tokens = [stemmer.stem(t) for t in tokens]                    # 4
        resultado.append(tokens)
    return resultado
reviews = ['Adorei o produto, entrega rapidíssima!',
           'Não gostei, veio quebrado e atrasado.']
print(pipeline_nlp(reviews))
# [['ador', 'produt', 'entreg', 'rap'],
#  ['gost', 'vei', 'quebr', 'atras']]